# Multimodal Fusion: Feature Concatenation Baseline

This notebook implements the **Feature Concatenation** multimodal baseline for in-hospital mortality prediction.
- **Temporal Modality**: 64-dimensional sequence embeddings extracted from the Bidirectional LSTM encoder ($t \in \mathbb{R}^{64}$).
- **Static Tabular Modality**: 64-dimensional clinical embeddings ($s \in \mathbb{R}^{64}$).
- **Concatenated Representation**: $[t \,\|\, s] \in \mathbb{R}^{128}$.

### Workflow
1. **Modal Alignment**: Align patient IDs across modalities, verify label consistency, and export fused archives.
2. **Standardization**: Fit $Z$-score scaler on training set only and normalize all splits.
3. **Downstream Classifier**: Train an MLP with class-weighted binary cross-entropy loss.
4. **Validation Tuning**: Select optimal decision threshold by maximizing validation F1 score.
5. **Evaluation**: Perform single-pass test evaluation and export comprehensive metrics.


## Part 1: Modal Alignment & Fused Matrix Generation


### Step 1: Imports & Workspace Configuration
Set up paths, project root resolution, and output directories.


In [3]:
from pathlib import Path
import numpy as np

# Resilient project root resolution (works from root or subfolder)
_start = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [_start, *_start.parents] if (p / "checkpoints").exists() and (p / "processed").exists()), _start)
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
TABULAR_DIR = CHECKPOINT_DIR
OUTPUT_DIR = CHECKPOINT_DIR / "fusion"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)



### Step 2: Patient Alignment & Fused Embedding Generation
Match patient IDs between temporal LSTM and static tabular modalities, verify consistency, concatenate into $(N, 128)$, and export `fused_embeddings_{split}.npz`.


In [4]:
for split in ("train", "val", "test"):
    print(f"\nProcessing {split} split...")

    # Load LSTM embeddings, patient IDs, and labels
    temporal = np.load(
        CHECKPOINT_DIR / f"temporal_embeddings_{split}.npy"
    )
    temporal_ids = np.load(
        CHECKPOINT_DIR / f"temporal_patient_ids_{split}.npy"
    ).reshape(-1).astype(np.int64)
    temporal_labels = np.load(
        CHECKPOINT_DIR / f"temporal_labels_{split}.npy"
    ).reshape(-1)

    # Prefer split-specific tabular embeddings.
    split_path = TABULAR_DIR / f"tabular_embeddings_{split}.npz"
    all_path = TABULAR_DIR / "tabular_embeddings_all.npz"

    if split_path.exists():
        tabular_path = split_path
    elif all_path.exists():
        tabular_path = all_path
    else:
        raise FileNotFoundError(
            f"Could not find {split_path} or {all_path}. "
            "Check TABULAR_DIR and your filenames."
        )

    data = np.load(tabular_path)
    tabular = data["embeddings"]
    tabular_ids = data["patient_ids"].reshape(-1).astype(np.int64)
    tabular_labels = data["labels"].reshape(-1)

    # Verify shapes and data quality
    assert temporal.ndim == 2 and temporal.shape[1] == 64, (
        f"Unexpected temporal shape: {temporal.shape}"
    )
    assert tabular.ndim == 2 and tabular.shape[1] == 64, (
        f"Unexpected tabular shape: {tabular.shape}"
    )
    assert len(temporal) == len(temporal_ids) == len(temporal_labels)
    assert len(tabular) == len(tabular_ids) == len(tabular_labels)
    assert len(np.unique(temporal_ids)) == len(temporal_ids)
    assert len(np.unique(tabular_ids)) == len(tabular_ids)
    assert np.isfinite(temporal).all(), "Temporal embeddings contain NaN/Inf."
    assert np.isfinite(tabular).all(), "Tabular embeddings contain NaN/Inf."

    # Map each patient ID to its row in the tabular embeddings.
    tabular_row = {int(pid): i for i, pid in enumerate(tabular_ids)}

    missing_ids = [
        int(pid) for pid in temporal_ids
        if int(pid) not in tabular_row
    ]
    if missing_ids:
        raise ValueError(
            f"{split}: {len(missing_ids)} LSTM patients have no "
            f"tabular embedding. Examples: {missing_ids[:5]}"
        )

    # Reorder tabular vectors to match LSTM patient order.
    indices = np.array(
        [tabular_row[int(pid)] for pid in temporal_ids],
        dtype=np.int64
    )
    tabular_aligned = tabular[indices]
    labels_aligned = tabular_labels[indices]

    # Confirm both modalities use the same mortality labels.
    if not np.array_equal(temporal_labels, labels_aligned):
        raise ValueError(
            f"{split}: LSTM and tabular labels don't match."
        )

    # Concatenate: (N, 64) + (N, 64) -> (N, 128)
    fused = np.concatenate(
        [temporal, tabular_aligned],
        axis=1
    )

    assert fused.shape == (len(temporal_ids), 128)
    assert np.isfinite(fused).all()

    # Save the fused matrix and its matching metadata.
    np.save(
        OUTPUT_DIR / f"fused_embeddings_{split}.npy",
        fused
    )
    np.savez_compressed(
        OUTPUT_DIR / f"fused_embeddings_{split}.npz",
        patient_ids=temporal_ids,
        labels=temporal_labels,
        temporal_embeddings=temporal,
        tabular_embeddings=tabular_aligned,
        fused_embeddings=fused
    )

    print(f"Patients: {len(temporal_ids):,}")
    print(f"Temporal shape: {temporal.shape}")
    print(f"Tabular shape:  {tabular_aligned.shape}")
    print(f"Fused shape:    {fused.shape}")
    print("All fused embeddings finite:", np.isfinite(fused).all())

print("\nFusion complete.")
print("Output folder:", OUTPUT_DIR.resolve())



Processing train split...
Patients: 153,831
Temporal shape: (153831, 64)
Tabular shape:  (153831, 64)
Fused shape:    (153831, 128)
All fused embeddings finite: True

Processing val split...
Patients: 24,186
Temporal shape: (24186, 64)
Tabular shape:  (24186, 64)
Fused shape:    (24186, 128)
All fused embeddings finite: True

Processing test split...
Patients: 22,809
Temporal shape: (22809, 64)
Tabular shape:  (22809, 64)
Fused shape:    (22809, 128)
All fused embeddings finite: True

Fusion complete.
Output folder: /Users/aryandwivedi/Desktop/Project Hail Minor/checkpoints/fusion


## Part 2: Downstream Classifier Training & Evaluation


### Step 3: Classifier Hyperparameters & Hardware Setup
Configure batch size, learning rate, random seed (42), and target device (CUDA / MPS / CPU).


In [5]:
from pathlib import Path
import copy
import json
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_recall_curve,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

# 1. Configuration
SEED = 42
BATCH_SIZE = 512
MAX_EPOCHS = 50
PATIENCE = 7
LR = 1e-3

torch.manual_seed(SEED)
np.random.seed(SEED)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

_start = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [_start, *_start.parents] if (p / "checkpoints").exists() and (p / "processed").exists()), _start)
FUSION_DIR = PROJECT_ROOT / "checkpoints" / "fusion"
OUT_DIR = FUSION_DIR / "concat_baseline"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Device:", device)
print("Fusion directory:", FUSION_DIR.resolve())




Device: mps
Fusion directory: /Users/aryandwivedi/Desktop/Project Hail Minor/checkpoints/fusion


### Step 4: Load Fused Cohorts & Validate Split Disjointness
Load fused 128-d embeddings for train, validation, and test splits; ensure zero patient leakage across splits.


In [6]:
# 2. Load the three splits

def load_split(split):
    path = FUSION_DIR / f"fused_embeddings_{split}.npz"

    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")

    with np.load(path) as data:
        X = data["fused_embeddings"].astype(np.float32)
        ids = data["patient_ids"].reshape(-1).astype(np.int64)
        y = data["labels"].reshape(-1).astype(np.float32)

    assert X.ndim == 2 and X.shape[1] == 128, X.shape
    assert len(X) == len(ids) == len(y)
    assert len(np.unique(ids)) == len(ids), (
        f"Duplicate patient IDs in {split}"
    )
    assert np.isfinite(X).all(), (
        f"Invalid embedding values in {split}"
    )
    assert np.isfinite(y).all()
    assert set(np.unique(y)).issubset({0.0, 1.0})

    return X, y, ids


X_train, y_train, ids_train = load_split("train")
X_val, y_val, ids_val = load_split("val")
X_test, y_test, ids_test = load_split("test")

assert not set(ids_train) & set(ids_val)
assert not set(ids_train) & set(ids_test)
assert not set(ids_val) & set(ids_test)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)




Train: (153831, 128)
Validation: (24186, 128)
Test: (22809, 128)


### Step 5: Feature Standardization (Train Statistics Only)
Fit mean and standard deviation on the training split only to prevent data leakage, then transform train, val, and test splits.


In [7]:
# 3. Standardize using TRAIN data only
# The temporal and tabular embedding dimensions may
# have different numerical scales.

mean = X_train.mean(axis=0)
std = X_train.std(axis=0)
std[std < 1e-8] = 1.0

X_train = ((X_train - mean) / std).astype(np.float32)
X_val = ((X_val - mean) / std).astype(np.float32)
X_test = ((X_test - mean) / std).astype(np.float32)

np.savez(
    OUT_DIR / "input_scaler.npz",
    mean=mean,
    std=std,
)



### Step 6: PyTorch DataLoaders & Class Imbalance Weighting
Construct batch loaders (batch size 512) and compute positive-class loss weight ($w = N_{\text{neg}} / N_{\text{pos}}$).


In [8]:
# 4. Create data loaders
def make_loader(X, y, shuffle):
    dataset = TensorDataset(
        torch.from_numpy(X),
        torch.from_numpy(y),
    )
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=0,
    )

train_loader = make_loader(X_train, y_train, True)
val_loader = make_loader(X_val, y_val, False)
test_loader = make_loader(X_test, y_test, False)




### Step 7: Fusion Classifier Model Architecture
Define a 2-layer MLP ($128 \to 64 \to 1$) with ReLU activation and 25% dropout, initialize loss and optimizer, and create prediction helper.


In [9]:
# 5. Define the downstream prediction model
class FusionClassifier(nn.Module):
    def __init__(self, input_dim=128, dropout=0.25):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        return self.network(x).squeeze(-1)


model = FusionClassifier().to(device)

n_pos = int((y_train == 1).sum())
n_neg = int((y_train == 0).sum())

criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(
        n_neg / max(n_pos, 1),
        dtype=torch.float32,
        device=device,
    )
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LR,
    weight_decay=1e-4,
)

print(f"Training positives: {n_pos:,}")
print(f"Training negatives: {n_neg:,}")

# 6. Prediction helper
def predict_probabilities(loader):
    model.eval()
    labels, probabilities = [], []

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)

            logits = model(xb)
            probs = torch.sigmoid(logits)

            labels.append(yb.numpy())
            probabilities.append(probs.cpu().numpy())

    return (
        np.concatenate(labels),
        np.concatenate(probabilities),
    )



Training positives: 13,634
Training negatives: 140,197


### Step 8: Training Loop with Validation AUPRC Early Stopping
Train for up to 50 epochs with patience = 7. Select model checkpoint based on highest validation AUPRC.


In [10]:
# 7. Train; select checkpoint by validation AUPRC
best_val_ap = -np.inf
best_state = None
wait = 0
history = []

for epoch in range(MAX_EPOCHS):
    model.train()
    total_loss = 0.0
    total_samples = 0

    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad(set_to_none=True)

        logits = model(xb)
        loss = criterion(logits, yb)

        loss.backward()
        nn.utils.clip_grad_norm_(
            model.parameters(), max_norm=1.0
        )
        optimizer.step()

        total_loss += loss.item() * len(yb)
        total_samples += len(yb)

    train_loss = total_loss / total_samples

    yv, pv = predict_probabilities(val_loader)
    val_ap = average_precision_score(yv, pv)
    val_auc = roc_auc_score(yv, pv)

    history.append({
        "epoch": epoch + 1,
        "train_loss": float(train_loss),
        "val_auprc": float(val_ap),
        "val_auroc": float(val_auc),
    })

    print(
        f"Epoch {epoch + 1:02d}/{MAX_EPOCHS} | "
        f"Train loss={train_loss:.4f} | "
        f"Val AUPRC={val_ap:.4f} | "
        f"Val AUROC={val_auc:.4f}"
    )

    if val_ap > best_val_ap:
        best_val_ap = val_ap
        best_state = copy.deepcopy(model.state_dict())
        wait = 0
    else:
        wait += 1
        if wait >= PATIENCE:
            print("Early stopping.")
            break

if best_state is None:
    raise RuntimeError("No best model checkpoint was selected.")

model.load_state_dict(best_state)

torch.save(
    model.state_dict(),
    OUT_DIR / "concat_classifier.pt",
)

with open(OUT_DIR / "training_history.json", "w") as f:
    json.dump(history, f, indent=2)

print(f"Best validation AUPRC: {best_val_ap:.4f}")




Epoch 01/50 | Train loss=0.7348 | Val AUPRC=0.5163 | Val AUROC=0.8741
Epoch 02/50 | Train loss=0.7036 | Val AUPRC=0.5219 | Val AUROC=0.8761
Epoch 03/50 | Train loss=0.6996 | Val AUPRC=0.5208 | Val AUROC=0.8758
Epoch 04/50 | Train loss=0.6961 | Val AUPRC=0.5203 | Val AUROC=0.8758
Epoch 05/50 | Train loss=0.6945 | Val AUPRC=0.5223 | Val AUROC=0.8777
Epoch 06/50 | Train loss=0.6915 | Val AUPRC=0.5261 | Val AUROC=0.8777
Epoch 07/50 | Train loss=0.6921 | Val AUPRC=0.5240 | Val AUROC=0.8769
Epoch 08/50 | Train loss=0.6901 | Val AUPRC=0.5262 | Val AUROC=0.8785
Epoch 09/50 | Train loss=0.6892 | Val AUPRC=0.5260 | Val AUROC=0.8774
Epoch 10/50 | Train loss=0.6873 | Val AUPRC=0.5268 | Val AUROC=0.8787
Epoch 11/50 | Train loss=0.6877 | Val AUPRC=0.5234 | Val AUROC=0.8775
Epoch 12/50 | Train loss=0.6852 | Val AUPRC=0.5282 | Val AUROC=0.8786
Epoch 13/50 | Train loss=0.6844 | Val AUPRC=0.5265 | Val AUROC=0.8790
Epoch 14/50 | Train loss=0.6837 | Val AUPRC=0.5246 | Val AUROC=0.8771
Epoch 15/50 | Train 

### Step 9: Decision Threshold Tuning (Validation F1)
Sweep precision-recall curve on the validation set to find the classification threshold that maximizes F1 score.


In [11]:
# 8. Select threshold using VALIDATION F1

yv, pv = predict_probabilities(val_loader)

precision_curve, recall_curve, thresholds = (
    precision_recall_curve(yv, pv)
)

f1_curve = (
    2 * precision_curve[:-1] * recall_curve[:-1]
    / (precision_curve[:-1] + recall_curve[:-1] + 1e-12)
)

best_idx = int(np.argmax(f1_curve))
best_threshold = float(thresholds[best_idx])

print(f"Selected validation threshold: {best_threshold:.4f}")
print(f"Validation F1: {f1_curve[best_idx]:.4f}")




Selected validation threshold: 0.7802
Validation F1: 0.5035


### Step 10: Held-Out Test Evaluation & Metrics Export
Evaluate the held-out test cohort once using the tuned decision threshold. Compute AUROC, AUPRC, Accuracy, F1, Confusion Matrix, and export `test_metrics.json`.


In [12]:
# 9. Evaluate TEST SET ONCE
yt, pt = predict_probabilities(test_loader)
yt_pred = (pt >= best_threshold).astype(int)

results = {
    "fusion": "concatenation",
    "classifier": "MLP",
    "threshold": best_threshold,
    "test_auroc": float(roc_auc_score(yt, pt)),
    "test_auprc": float(average_precision_score(yt, pt)),
    "test_accuracy": float(accuracy_score(yt, yt_pred)),
    "test_balanced_accuracy": float(
        balanced_accuracy_score(yt, yt_pred)
    ),
    "test_precision": float(
        precision_score(yt, yt_pred, zero_division=0)
    ),
    "test_recall": float(
        recall_score(yt, yt_pred, zero_division=0)
    ),
    "test_f1": float(
        f1_score(yt, yt_pred, zero_division=0)
    ),
    "confusion_matrix": confusion_matrix(
        yt, yt_pred, labels=[0, 1]
    ).tolist(),
}

print("\nCONCATENATION BASELINE")
for key, value in results.items():
    print(f"{key}: {value}")

with open(OUT_DIR / "test_metrics.json", "w") as f:
    json.dump(results, f, indent=2)

print("\nSaved results to:", OUT_DIR.resolve())



CONCATENATION BASELINE
fusion: concatenation
classifier: MLP
threshold: 0.7802372574806213
test_auroc: 0.8733322463575262
test_auprc: 0.5268600489358062
test_accuracy: 0.9032399491428822
test_balanced_accuracy: 0.7316964278262592
test_precision: 0.4725848563968668
test_recall: 0.5218644882268141
test_f1: 0.49600365380223793
confusion_matrix: [[19516, 1212], [995, 1086]]

Saved results to: /Users/aryandwivedi/Desktop/Project Hail Minor/checkpoints/fusion/concat_baseline
